# Mission : Pilotez l'atterrisseur lunaire Eagle-1


## Installation des dépendances

Ce notebook est exécuté sur Google Colab, la cellule suivante installe les dépendances nécessaires.

In [ ]:
!pip install gymnasium[box2d] stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.1 MB/s eta 0:00:00


## Création de l'environnement

In [ ]:
import gymnasium as gym

# Create environment
env = gym.make("LunarLander-v3")
# env = gym.make("LunarLander-v3", render_mode="rgb_array")


print(f"Observation space: {env.observation_space}")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Observation sample: {env.observation_space.sample()}")

print(f"Action space: {env.action_space}")
print(f"Action sample: {env.action_space.sample()}")
print(f"Nombre d'actions: {env.action_space.n}")

Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Observation shape: (8,)
Observation sample: [ 1.2035573  -1.4762939   7.037694   -0.28728837 -6.1089263  -2.5333083
  0.12325706  0.7548049 ]
Action space: Discrete(4)
Action sample: 1
Nombre d'actions: 4


**Observation de l'environnement**

L'espace d'observation contient 8 variables continues : position (x, y),
vitesse (x, y), angle, vitesse angulaire, et 2 capteurs de contact (booléens).

L'espace d'action est discret avec 4 actions possibles :
- 0 : ne rien faire
- 1 : allumer le propulseur gauche
- 2 : allumer le propulseur principal (bas)
- 3 : allumer le propulseur droit

## Choix de l'algorithme

L'espace d'action de LunarLander-v3 est discret (4 actions), ce qui oriente le choix vers deux algorithmes compatibles avec Stable-Baselines3.

**DQN (Deep Q-Network)** apprend une fonction Q(s, a) qui estime la récompense future pour chaque action depuis un état donné. Il sélectionne toujours l'action avec la valeur Q la plus élevée et stocke ses expériences dans un replay buffer pour stabiliser l'apprentissage (off-policy).

**PPO (Proximal Policy Optimization)** apprend directement une politique : une distribution de probabilités sur les actions à prendre. Il ajoute une contrainte de clipping pour éviter des mises à jour trop brusques (on-policy). Bien que conçu pour les espaces continus, il fonctionne aussi sur du discret.

On part sur **DQN** pour cette mission : il est explicitement recommandé par le brief pour les espaces d'action discrets, et a déjà été appliqué sur CartPole en exercice 3, le pipeline est donc connu.

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy


def create_model_DQN(policy, env, **kwargs):
    model = DQN(policy, env, verbose=1, tensorboard_log="./logs/", **kwargs)
    return model


def learn_and_evaluated_model(model, env, total_timesteps=500000, n_eval_episodes=50):
    model.learn(total_timesteps)
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes)
    print(f"Récompense moyenne : {mean_reward:.2f} +/- {std_reward:.2f}")
    return mean_reward, std_reward

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
model_baseline = create_model_DQN('MlpPolicy', env)
learn_and_evaluated_model(model_baseline, env)

model_baseline.save("dqn_baseline")
print("Modèle sauvegardé : dqn_baseline.zip")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 81       |
|    ep_rew_mean      | -209     |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 516      |
|    time_elapsed     | 0        |
|    total_timesteps  | 324      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 2.61     |
|    n_updates        | 55       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85.4     |
|    ep_rew_mean      | -126     |
|    exploration_rate | 0.987    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 531      |
|    time_elapsed     | 1        |
|    total_timesteps  | 683      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.83     |
|    n_updates      

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Récompense moyenne : 37.68 +/- 125.08
Modèle sauvegardé : dqn_baseline.zip


**Modèle baseline**

Entraînement : 500 000 timesteps, hyperparamètres par défaut SB3.
Récompense moyenne : 37.68 +/- 125.08 sur 50 épisodes.

L'écart-type élevé (125.08) traduit une forte variabilité : l'agent réussit certains atterrissages mais s'écrase ou sort des limites sur d'autres. Le score moyen reste loin de l'objectif de 200. L'optimisation des hyperparamètres est nécessaire.

## Optimisation des hyperparamètres

La stratégie : modifier un seul hyperparamètre à la fois, évaluer sur 50 épisodes, puis passer au suivant en conservant les changements bénéfiques.

Paramètres explorés :

`learning_starts` : nombre de steps aléatoires avant que l'agent commence à apprendre. Le défaut SB3 est 50 000, soit 10 % du budget total à ne rien apprendre. On le réduit à 1 000.

`learning_rate` : vitesse à laquelle le réseau corrige ses poids après chaque erreur. Trop élevé : instabilité. Trop faible : convergence lente. Défaut SB3 : 1e-4.

`batch_size` : taille de l'échantillon tiré du buffer à chaque mise à jour. Un batch trop petit produit un signal bruité. Défaut SB3 : 32.

`target_update_interval` : fréquence de synchronisation du réseau cible. DQN maintient deux réseaux, un actif (qui apprend) et un cible (référence stable pour calculer les erreurs). Défaut SB3 : 10 000.

`buffer_size` : capacité de la mémoire d'expériences. 1 million par défaut est disproportionné pour 500k steps. On réduit à 50 000.

### Expérience 1 (learning_starts : 50 000 → 10 000)


In [ ]:
model_exp1 = create_model_DQN('MlpPolicy', env, learning_starts=10000)
learn_and_evaluated_model(model_exp1, env)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_2
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85.2     |
|    ep_rew_mean      | -126     |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 5648     |
|    time_elapsed     | 0        |
|    total_timesteps  | 341      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 78.8     |
|    ep_rew_mean      | -151     |
|    exploration_rate | 0.988    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 4848     |
|    time_elapsed     | 0        |
|    total_timesteps  | 630      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 83.6     |
|    ep

(np.float64(132.2197736956422), np.float64(126.97501928591682))

**Observation :**

Résultat : 132.22 +/- 126.98 sur 50 épisodes.

Amélioration significative par rapport au baseline (37.68). 
En réduisant `learning_starts` de 50 000 à 10 000, l'agent dispose de 40 000 steps supplémentaires d'apprentissage effectif au lieu d'exploration aléatoire. Sur un budget de 500 000 steps, ce gain de temps est déterminant. 

L'écart-type reste élevé (126.98), signe que la politique est encore instable : avec un taux d'apprentissage de 1e-4 par défaut, les mises à jour sont trop timides pour profiter pleinement de ce démarrage anticipé. Prochain paramètre : `learning_rate`.

### Expérience 2 (learning_rate : 1e-4 → 6.3e-4) 


In [ ]:
model_exp2 = create_model_DQN(
    'MlpPolicy', env,
    learning_starts=10000,
    learning_rate=6.3e-4,
)
learn_and_evaluated_model(model_exp2, env)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_3
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85.2     |
|    ep_rew_mean      | -188     |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 5216     |
|    time_elapsed     | 0        |
|    total_timesteps  | 341      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 86.1     |
|    ep_rew_mean      | -172     |
|    exploration_rate | 0.987    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 5558     |
|    time_elapsed     | 0        |
|    total_timesteps  | 689      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 86.5     |
|    ep

(np.float64(-37.0653433103417), np.float64(49.24167208063135))

**Observation :**

Résultat : -37.07 +/- 49.24 sur 50 épisodes.

Régression importante. Multiplier le taux d'apprentissage par 6 sans ajuster les autres paramètres du buffer est contre-productif : avec `batch_size=32` et `target_update_interval=10 000` par défaut, le signal d'apprentissage est bruité et le réseau cible rarement synchronisé. 

Un LR élevé amplifie ce bruit et fait diverger les valeurs Q. L'écart-type faible (49.24) indique que l'agent converge vers un comportement systématiquement mauvais plutôt que d'alterner entre bons et mauvais épisodes. La solution n'est pas d'abandonner ce LR mais de stabiliser son environnement d'apprentissage : c'est l'objet de l'expérience 3.

### Expérience 3 (batch_size, target_update_interval, buffer_size)

Ces trois paramètres gouvernent la mécanique du replay buffer : la taille des lots d'apprentissage, la fréquence de synchronisation du réseau cible, et la capacité mémoire. Ils sont regroupés car ils interagissent directement : augmenter le `batch_size` sans réduire le `target_update_interval` produit des effets contradictoires.

Changements : `batch_size` 32 → 128, `target_update_interval` 10 000 → 250, `buffer_size` 1 000 000 → 50 000.


In [ ]:
model_exp3 = create_model_DQN(
    'MlpPolicy', env,
    learning_starts=10000,
    learning_rate=6.3e-4,
    batch_size=128,
    target_update_interval=250,
    buffer_size=50000,
)
learn_and_evaluated_model(model_exp3, env)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_4
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 95.5     |
|    ep_rew_mean      | -138     |
|    exploration_rate | 0.993    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 5625     |
|    time_elapsed     | 0        |
|    total_timesteps  | 382      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 92.9     |
|    ep_rew_mean      | -148     |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 5771     |
|    time_elapsed     | 0        |
|    total_timesteps  | 743      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 95.6     |
|    ep

(np.float64(99.11008403353325), np.float64(83.18165474617707))

**Observation :**

Résultat : 99.11 +/- 83.18 sur 50 épisodes.

Récupération par rapport à l'expérience 2 (-37.07). Des batchs plus grands (128) fournissent un gradient plus stable, une synchronisation plus fréquente du réseau cible (toutes les 250 steps au lieu de 10 000) donne un signal d'erreur plus cohérent, et réduire le buffer à 50 000 force l'agent à réutiliser des expériences récentes plus pertinentes. 

Ces ajustements stabilisent l'effet du LR élevé. Le score (99.11) reste en dessous de l'expérience 1 (132.22), ce qui indique que 500 000 steps ne suffisent pas encore pour que cette configuration converge pleinement. Prochain test : doubler la durée d'entraînement.

## Modèle final

In [ ]:
model_final = create_model_DQN(
    'MlpPolicy', env,
    learning_starts=1000,
    learning_rate=6.3e-4,
    batch_size=128,
    target_update_interval=250,
    buffer_size=50000,
)
mean_reward, std_reward = learn_and_evaluated_model(
    model_final, env,
    total_timesteps=1_000_000,
    n_eval_episodes=100,
)

model_final.save("dqn_final")
print("Modèle sauvegardé : dqn_final.zip")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
|    loss             | 1.19     |
|    n_updates        | 120680   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 415      |
|    ep_rew_mean      | 206      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1344     |
|    fps              | 909      |
|    time_elapsed     | 534      |
|    total_timesteps  | 486071   |
| train/              |          |
|    learning_rate    | 0.00063  |
|    loss             | 0.495    |
|    n_updates        | 121267   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 426      |
|    ep_rew_mean      | 203      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1348     |
|    fps              | 909      |
|    time_el

**Observation :**

Résultat : 217.07 +/- 76.20 sur 100 épisodes.

Objectif atteint (≥ 200). Avec 1 million de steps, la configuration de l'expérience 3 a le temps de converger : le LR élevé accélère l'apprentissage, les paramètres du buffer stabilisent les mises à jour, et la durée prolongée permet à la politique de se consolider. 

L'écart-type (76.20) a nettement baissé par rapport aux expériences précédentes, signe que l'agent atterrit de manière cohérente plutôt qu'aléatoire. Le modèle est sauvegardé sous `dqn_final.zip`.

In [ ]:
from google.colab import files
import zipfile

with zipfile.ZipFile("dqn_models.zip", "w") as zf:
    zf.write("dqn_baseline.zip")
    zf.write("dqn_final.zip")

files.download("dqn_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>